# Thesis Analysis

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "thesis_experiment").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from notebooks.analysis_tools import (
    APTOS_FONT_DIR,
    DEFAULT_RUN_SPECS,
    PLOT_CONDITION_ORDER,
    build_analysis_context,
    build_chunk_dashboard_tables,
    build_steps_summary_table,
    configure_matplotlib_font,
    plot_chunk_dashboard_v3,
    plot_episode_cost_dashboard_v3,
    plot_error_dashboard_v3,
    plot_steps_dashboard_v8,
    plot_task_condition_success_heatmap_light_continuous,
    plot_task_difficulty_v3,
    summarize_analysis_context,
)


In [ ]:
SAVE_FIGURES = True
FIG_DIR = REPO_ROOT / "thesis_experiment" / "generated_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

RUN_SPECS = DEFAULT_RUN_SPECS
TASK_COL = "task_id"
COND_COL = "plot_condition"

FONT_DIR = APTOS_FONT_DIR
FONT_FAMILY = configure_matplotlib_font(FONT_DIR)
print(f"Matplotlib font family: {FONT_FAMILY}")

TEXT_STYLE = {
    "suptitle_fontsize": 30,
    "panel_title_fontsize": 24,
    "axis_label_fontsize": 22,
    "xtick_fontsize": 20,
    "ytick_fontsize": 20,
    "legend_fontsize": 18,
    "legend_title_fontsize": 18,
}

ANNOTATION_STYLE = {
    "annotation_fontsize": 18,
    "binary_label_fontsize": 18,
    "rate_label_fontsize": 18,
    "action_share_label_fontsize": 17,
    "cell_fontsize": 16,
}

COMMON_PLOT_STYLE = {
    "xlabels_two_lines": True,
    "xtick_rotation": 0,
    "xtick_pad": 14,
    "grid_alpha": 0.25,
    "save_dpi": 350,
}

FIGURE_STYLE = {
    "heatmap_figsize": (16.0, 9.2),
    "single_plot_figsize": (16.0, 9.2),
    "dashboard_1x2_figsize": (18.0, 9.6),
    "dashboard_2x2_figsize": (19.0, 14.4),
}


In [ ]:
FIGURE_STYLE["dashboard_2x2_figsize"]

In [ ]:
context = build_analysis_context(run_specs=RUN_SPECS)
df_all = context["df_all"]
run_inventory = context["run_inventory"]
chunks_piv = context["chunks_piv"]
steps_piv = context["steps_piv"]

display(run_inventory)
print("Total rows:", len(df_all))
print("Conditions:", ", ".join(PLOT_CONDITION_ORDER))


In [ ]:
summary = summarize_analysis_context(context)

display(pd.DataFrame([summary["completeness"]]))
display(summary["chunk_quality_comparison"])
display(chunks_piv)
display(steps_piv)


## Task-condition heatmap


In [ ]:
HEATMAP_PARAMS = {
    "cond_col": COND_COL,
    "outpath_png": str(FIG_DIR / "heatmap_success_task_condition_light_continuous.png") if SAVE_FIGURES else None,
    "annotate": True,
    "figsize": FIGURE_STYLE["heatmap_figsize"],
    "task_id_chars": 22,
    "cell_fontsize": ANNOTATION_STYLE["cell_fontsize"],
    "ytick_fontsize": TEXT_STYLE["ytick_fontsize"],
    "xtick_fontsize": TEXT_STYLE["xtick_fontsize"],
    "axis_label_fontsize": TEXT_STYLE["axis_label_fontsize"],
    "xlabels_two_lines": COMMON_PLOT_STYLE["xlabels_two_lines"],
    "title": "",
    "title_fontsize": TEXT_STYLE["panel_title_fontsize"],
    "annotation_as_percent": True,
    "colorbar_label": "Success rate",
    "colorbar_label_fontsize": TEXT_STYLE["axis_label_fontsize"],
    "colorbar_tick_fontsize": TEXT_STYLE["ytick_fontsize"],
    "left_margin": 0.08,
    "right_margin": 0.95,
    "bottom_margin": 0.20,
    "top_margin": 0.93,
    "include_mean_row": True,
    "mean_row_label": "Mean",
}

fig_heatmap, ax_heatmap, heatmap_pivot = plot_task_condition_success_heatmap_light_continuous(
    df_all,
    **HEATMAP_PARAMS,
)
plt.show()


## Task difficulty


In [ ]:
TASK_DIFFICULTY_PARAMS = {
    "task_col": TASK_COL,
    "fig_width": FIGURE_STYLE["single_plot_figsize"][0],
    "row_height": 0.72,
    "min_fig_height": FIGURE_STYLE["single_plot_figsize"][1],
    "tight_layout_rect": (0, 0, 1, 0.98),
    "task_label_max_chars": 36,
    "sort_by": ("success_rate", "duration_mean"),
    "sort_ascending": (True, False),
    "main_blue": "#56B4E9",
    "contrast_orange": "#D55E00",
    "bar_edgecolor": "black",
    "suptitle": None,
    "suptitle_fontsize": TEXT_STYLE["suptitle_fontsize"],
    "panel_title":None,
    "panel_title_fontsize": TEXT_STYLE["panel_title_fontsize"],
    "axis_label_fontsize": TEXT_STYLE["axis_label_fontsize"],
    "xtick_fontsize": TEXT_STYLE["xtick_fontsize"],
    "ytick_fontsize": TEXT_STYLE["ytick_fontsize"],
    "annotation_fontsize": ANNOTATION_STYLE["annotation_fontsize"],
    "bar_height": 0.72,
    "bar_linewidth": 0.6,
    "bar_alpha": 1.0,
    "grid_alpha": COMMON_PLOT_STYLE["grid_alpha"],
    "xlim": (0.0, 1.10),
    "show_reference_line": False,
    "reference_line_value": None,
    "reference_line_linestyle": "--",
    "reference_line_linewidth": 1.5,
    "reference_line_alpha": 0.9,
    "show_bar_values": True,
    "bar_value_fmt": "{:.0%}",
    "bar_value_dx": 0.015,
    "outpath_png": str(FIG_DIR / "task_difficulty_v3.png") if SAVE_FIGURES else None,
    "save_dpi": COMMON_PLOT_STYLE["save_dpi"],
}

fig_td, ax_td, task_profile = plot_task_difficulty_v3(
    df_all,
    **TASK_DIFFICULTY_PARAMS,
)
display(task_profile)
plt.show()


## Episode cost dashboard


In [ ]:
COST_PARAMS = {
    "cond_col": COND_COL,
    "outpath_png": str(FIG_DIR / "episode_cost_dashboard_v3.png") if SAVE_FIGURES else None,
    "title": None,
    "figsize": (21.0, 9.6),
    "wspace": 0.28,
    "hspace": 0.20,
    "point_size": 55,
    "point_alpha": 0.35,
    "point_edgecolor": "black",
    "point_linewidth": 0.5,
    "mean_marker": "D",
    "mean_marker_size": 220,
    "mean_alpha": 1.0,
    "mean_edgecolor": "black",
    "mean_linewidth": 0.9,
    "err_capsize": 5,
    "err_linewidth": 1.2,
    "jitter": 0.10,
    "random_seed": 42,
    "main_blue": "#56B4E9",
    "contrast_orange": "#D55E00",
    "blue_light": "#8ECAE6",
    "blue_dark": "#1F77B4",
    "fallback_cmap_name": "tab10",
    "suptitle_fontsize": TEXT_STYLE["suptitle_fontsize"],
    "panel_title_fontsize": TEXT_STYLE["panel_title_fontsize"],
    "axis_label_fontsize": TEXT_STYLE["axis_label_fontsize"],
    "xtick_fontsize": TEXT_STYLE["xtick_fontsize"],
    "ytick_fontsize": TEXT_STYLE["ytick_fontsize"],
    "legend_fontsize": TEXT_STYLE["legend_fontsize"],
    "legend_title_fontsize": TEXT_STYLE["legend_title_fontsize"],
    "suptitle_y": 0.98,
    "title_pad": 0,
    "title_y": 1.08,
    "show_legend": True,
    "legend_title": "Experiment",
    "legend_loc": "center left",
    "legend_bbox_to_anchor": (1.02, 0.5),
    "legend_frameon": False,
    "legend_ncol": 1,
    "legend_columnspacing": 1.2,
    "legend_handlelength": 1.6,
    "legend_handletextpad": 0.5,
    "legend_labelspacing": 0.6,
    "legend_borderaxespad": 0.0,
    "title_left": "(A) Average LLM requests per episode",
    "title_right": "(B) Average episode duration",
    "xlabel": "Experiment",
    "ylabel_left": "Average LLM requests per episode",
    "ylabel_right": "Average duration per episode (sec)",
    "xlabels_two_lines": COMMON_PLOT_STYLE["xlabels_two_lines"],
    "xtick_rotation": COMMON_PLOT_STYLE["xtick_rotation"],
    "xtick_pad": COMMON_PLOT_STYLE["xtick_pad"],
    "grid_alpha": COMMON_PLOT_STYLE["grid_alpha"],
    "save_dpi": COMMON_PLOT_STYLE["save_dpi"],
    "left_margin": 0.07,
    "right_margin": 0.84,
    "bottom_margin": 0.17,
    "top_margin": 0.91,
}

fig_cost, axes_cost, run_cost_tbl, cost_summary_tbl = plot_episode_cost_dashboard_v3(
    df_all,
    **COST_PARAMS,
)
display(cost_summary_tbl)
plt.show()


## Chunk dashboard


In [ ]:
CHUNK_TABLES = build_chunk_dashboard_tables(
    df_all,
    cond_col=COND_COL,
    task_col=TASK_COL,
    rate_mode="weighted",
)
display(CHUNK_TABLES["rc"])
display(CHUNK_TABLES["cnt"])
display(CHUNK_TABLES["dyn_mean"])


In [ ]:
CHUNK_PARAMS = {
    "outpath_png": str(FIG_DIR / "chunk_dashboard_v3.png") if SAVE_FIGURES else None,
    "title": None,
    "figsize": (20.0, 14.4),
    "width_ratios": (1.08, 1.18),
    "height_ratios": (1.08, 1.08),
    "hspace": 0.48,
    "wspace": 0.30,
    "bar_width": 0.58,
    "bar_edgecolor": "black",
    "bar_linewidth": 0.6,
    "err_capsize": 5,
    "scatter_size": 48,
    "scatter_alpha": 0.95,
    "scatter_edgecolor": "white",
    "scatter_linewidth": 0.6,
    "jitter_frac": 0.40,
    "main_blue": "#56B4E9",
    "contrast_orange": "#D55E00",
    "blue_light": "#8ECAE6",
    "blue_dark": "#1F77B4",
    "suptitle_fontsize": TEXT_STYLE["suptitle_fontsize"],
    "panel_title_fontsize": TEXT_STYLE["panel_title_fontsize"],
    "axis_label_fontsize": TEXT_STYLE["axis_label_fontsize"],
    "xtick_fontsize": TEXT_STYLE["xtick_fontsize"],
    "ytick_fontsize": TEXT_STYLE["ytick_fontsize"],
    "legend_fontsize": TEXT_STYLE["legend_fontsize"],
    "suptitle_y": 0.98,
    "title_pad": 0,
    "title_y": 1.18,
    "legend_outside": True,
    "legend_y": 1.05,
    "legend_ncol": 2,
    "legend_columnspacing": 1.2,
    "legend_handlelength": 1.6,
    "legend_handletextpad": 0.5,
    "legend_labelspacing": 0.6,
    "xlabels_two_lines": COMMON_PLOT_STYLE["xlabels_two_lines"],
    "compact_condition_labels": True,
    "xtick_rotation": COMMON_PLOT_STYLE["xtick_rotation"],
    "xtick_pad": COMMON_PLOT_STYLE["xtick_pad"],
    "grid_alpha": COMMON_PLOT_STYLE["grid_alpha"],
    "ypad_top": 0.06,
    "ylim_a_max": 1.15,
    "save_dpi": COMMON_PLOT_STYLE["save_dpi"],
    "cond_col": COND_COL,
    "task_col": TASK_COL,
    "chunk_rate_mode": "weighted",
    "binary_label_fontsize": ANNOTATION_STYLE["binary_label_fontsize"],
    "rate_label_fontsize": ANNOTATION_STYLE["rate_label_fontsize"],
    "show_quality_error_bars": False,
    "show_quality_bar_labels": True,
    "quality_bar_label_position": "base",
    "quality_bar_label_base_frac": 0.04,
    "left_margin": 0.07,
    "right_margin": 0.98,
    "bottom_margin": 0.13,
    "top_margin": 0.93,
}

fig_chunk, axes_chunk, chunk_tables = plot_chunk_dashboard_v3(
    df_all,
    **CHUNK_PARAMS,
)
plt.show()


## Step dashboard


In [ ]:
steps_summary_tbl = build_steps_summary_table(
    df_all,
    cond_col=COND_COL,
)
display(steps_summary_tbl)


In [ ]:
STEPS_PARAMS = {
    "cond_col": COND_COL,
    "outpath_png": str(FIG_DIR / "steps_dashboard_v8.png") if SAVE_FIGURES else None,
    "save_dpi": COMMON_PLOT_STYLE["save_dpi"],
    "figsize": (20.0, 14.4),
    "bar_width": 0.58,
    "hspace": 0.56,
    "wspace": 0.32,
    "axis_label_fontsize": TEXT_STYLE["axis_label_fontsize"],
    "xtick_fontsize": TEXT_STYLE["xtick_fontsize"],
    "ytick_fontsize": TEXT_STYLE["ytick_fontsize"],
    "panel_title_fontsize": TEXT_STYLE["panel_title_fontsize"],
    "title_pad": 0,
    "title_y": 1.18,
    "xlabels_two_lines": COMMON_PLOT_STYLE["xlabels_two_lines"],
    "compact_condition_labels": True,
    "y_headroom_frac": 0.10,
    "ylabel_C": "Mean action share",
    "ylabel_D": "Mean failure share",
    "legend_outside": True,
    "legend_fontsize": TEXT_STYLE["legend_fontsize"],
    "legend_ncol": 2,
    "legend_y": 1.08,
    "emphasize_D_edge": True,
    "binary_label_fontsize": ANNOTATION_STYLE["binary_label_fontsize"],
    "rate_label_fontsize": ANNOTATION_STYLE["rate_label_fontsize"],
    "action_share_label_fontsize": ANNOTATION_STYLE["action_share_label_fontsize"],
    "show_quality_error_bars": False,
    "annotate_action_shares": False,
    "left_margin": 0.06,
    "right_margin": 0.96,
    "bottom_margin": 0.14,
    "top_margin": 0.96,
}

fig_steps, axes_steps, failure_tbl = plot_steps_dashboard_v8(
    df_all,
    **STEPS_PARAMS,
)
plt.show()


## Recovery dashboard


In [ ]:
ERROR_PARAMS = {
    "cond_col": COND_COL,
    "run_col": "run_id",
    "outpath_png": str(FIG_DIR / "error_dashboard_v3.png") if SAVE_FIGURES else None,
    "title": None,
    "figsize": (25.0, 9.6),
    "width_ratios": (1.08, 1.18),
    "wspace": 0.24,
    "bar_width": 0.58,
    "bar_edgecolor": "black",
    "bar_linewidth": 0.6,
    "err_capsize": 5,
    "err_linewidth": 1.2,
    "main_blue": "#56B4E9",
    "blue_light": "#8ECAE6",
    "blue_dark": "#1F77B4",
    "suptitle_fontsize": TEXT_STYLE["suptitle_fontsize"],
    "panel_title_fontsize": TEXT_STYLE["panel_title_fontsize"],
    "axis_label_fontsize": TEXT_STYLE["axis_label_fontsize"],
    "xtick_fontsize": TEXT_STYLE["xtick_fontsize"],
    "ytick_fontsize": TEXT_STYLE["ytick_fontsize"],
    "legend_fontsize": TEXT_STYLE["legend_fontsize"],
    "suptitle_y": 0.98,
    "title_pad": 0,
    "title_y": 1.18,
    "legend_outside": True,
    "legend_y": 1.08,
    "legend_loc": "center",
    "legend_ncol": 2,
    "legend_columnspacing": 1.2,
    "legend_handlelength": 1.6,
    "legend_handletextpad": 0.5,
    "legend_labelspacing": 0.6,
    "title_A": "(A) Recovery opportunities",
    "title_B": "(B) Recovery rate",
    "ylabel_A": "Mean opportunities per episode",
    "ylabel_B": "Recovery rate $r_{rec}$",
    "xlabels_two_lines": COMMON_PLOT_STYLE["xlabels_two_lines"],
    "compact_condition_labels": True,
    "xtick_rotation": COMMON_PLOT_STYLE["xtick_rotation"],
    "xtick_pad": COMMON_PLOT_STYLE["xtick_pad"],
    "grid_alpha": COMMON_PLOT_STYLE["grid_alpha"],
    "ylim_rate": (0.0, 1.0),
    "save_dpi": COMMON_PLOT_STYLE["save_dpi"],
    "binary_label_fontsize": ANNOTATION_STYLE["binary_label_fontsize"],
    "left_margin": 0.07,
    "right_margin": 0.96,
    "bottom_margin": 0.17,
    "top_margin": 0.91,
}

fig_err, axes_err, recovery_tbl = plot_error_dashboard_v3(
    df_all,
    **ERROR_PARAMS,
)
display(recovery_tbl)
plt.show()
